In [3]:
import os
import pandas as pd
import numpy as np
!pip install optuna
import optuna
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

def load_financial_data():
    """
    Baixa os dados reais de Inadimplência de Cartão de Crédito (UCI)
    e faz o cache localmente para eficiência.
    """
    data_dir = "data"
    file_path = f"{data_dir}/credit_card_default.csv"

    # URL do dataset público no repositório da UCI
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"

    if not os.path.exists(data_dir):
        os.makedirs(data_dir)

    if os.path.exists(file_path):
        print(f"Carregando dados financeiros do cache local: {file_path}")
        df = pd.read_csv(file_path)
    else:
        print("Buscando dados da internet (UCI Repository)...")
        # O arquivo original é um XLS; o Pandas cuida da leitura
        df = pd.read_excel(url, header=1)
        df = df.rename(columns={'default payment next month': 'target'})
        df.to_csv(file_path, index=False)
        print("Download concluído e salvo no cache local.")

    return df

def build_preprocessor():
    """
    Constrói o processador de features garantindo que variáveis
    financeiras não sofram distorções com outliers (RobustScaler).
    """
    # Excluindo a coluna ID, pois não tem valor preditivo
    num_features = [
        'LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
        'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2',
        'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
    ]
    # Variáveis categóricas (pagamentos históricos e demográficos)
    cat_features = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

    num_transformer = Pipeline(steps=[
        ('scaler', RobustScaler()) # Ideal para dados monetários com outliers extremos
    ])

    cat_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_features),
            ('cat', cat_transformer, cat_features)
        ])

    return preprocessor

def objective(trial, X_train, y_train, preprocessor):
    """
    Função objetivo para a Otimização Bayesiana com Optuna.
    Busca maximizar a precisão média (PR-AUC) do modelo.
    """
    # Espaço de busca sugerido probabilisticamente pelo Optuna
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0), # Lida com desbalanceamento
        'use_label_encoder': False,
        'eval_metric': 'aucpr'
    }

    model = xgb.XGBClassifier(**param, random_state=42)

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Validação Cruzada Estratificada para manter proporção de inadimplentes
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    aucpr_scores = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        pipeline.fit(X_tr, y_tr)

        # Prever as probabilidades para a métrica PR-AUC
        y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
        score = average_precision_score(y_val, y_pred_proba)
        aucpr_scores.append(score)

    return np.mean(aucpr_scores)

def main():
    df = load_financial_data()

    X = df.drop(columns=['ID', 'target'])
    y = df['target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    preprocessor = build_preprocessor()

    print("\nIniciando Otimização Bayesiana com Optuna...")
    # Cria o estudo direcionado a maximizar a métrica
    study = optuna.create_study(direction="maximize")

    # Passando os dados reais para a função objetivo
    func_objetivo = lambda trial: objective(trial, X_train, y_train, preprocessor)
    study.optimize(func_objetivo, n_trials=20)

    print("\nMelhores hiperparâmetros encontrados:")
    print(study.best_params)

    # Treinamento do Modelo Final com os melhores parâmetros
    best_params = study.best_params
    best_params['use_label_encoder'] = False
    best_params['eval_metric'] = 'aucpr'

    final_model = xgb.XGBClassifier(**best_params, random_state=42)

    final_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', final_model)
    ])

    print("\nTreinando o modelo final...")
    final_pipeline.fit(X_train, y_train)

    # Avaliação de Negócio Executiva
    y_pred = final_pipeline.predict(X_test)
    y_pred_proba = final_pipeline.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)

    print("\n--- Resultados Financeiros (Conjunto de Teste) ---")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"PR-AUC (Foco na classe de Default): {pr_auc:.4f}")
    print("\nRelatório de Classificação Detalhado:")
    print(classification_report(y_test, y_pred))

    joblib.dump(final_pipeline, 'xgboost_credit_default_pipeline.pkl')
    print("\nPipeline serializado como 'xgboost_credit_default_pipeline.pkl'.")

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 24.7 MB/s eta 0:00:00
Buscando dados da internet (UCI Repository)...


[I 2026-06-30 23:03:54,063] A new study created in memory with name: no-name-d51b9643-f519-4ac4-b6bf-ed5ec6977a9b


Download concluído e salvo no cache local.

Iniciando Otimização Bayesiana com Optuna...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:03:54] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:03:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:03:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-06-30 23:04:01,435] Trial 0 finished with value: 0.5281461313663184 and parameters: {'n_estimators': 139, 'max_depth': 10, 'learning_rate': 0.09378444802913026, 'subsample': 0.6628085460747241, 'colsample_bytree': 0.6794962242796649, 'scale_pos_weight': 4.421914228609634}. Best is trial


Melhores hiperparâmetros encontrados:
{'n_estimators': 164, 'max_depth': 9, 'learning_rate': 0.010612303888262122, 'subsample': 0.8135413695493716, 'colsample_bytree': 0.6803502210657584, 'scale_pos_weight': 2.9098467502423677}

Treinando o modelo final...

--- Resultados Financeiros (Conjunto de Teste) ---
ROC-AUC: 0.7784
PR-AUC (Foco na classe de Default): 0.5507

Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

           0       0.87      0.87      0.87      4673
           1       0.54      0.53      0.54      1327

    accuracy                           0.80      6000
   macro avg       0.71      0.70      0.70      6000
weighted avg       0.80      0.80      0.80      6000


Pipeline serializado como 'xgboost_credit_default_pipeline.pkl'.
